In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from finnews_sentiment.features.preprocess import clean_text, preprocess_text, prepare_dataset

In [2]:
df = pd.read_csv("data/financial_sentiment.csv")
df = df.rename(columns={"Sentence": "text", "Sentiment": "sentiment"})
print(df.shape)

(5842, 2)


Проверяю очистку руками на нескольких примерах

In [3]:
for t in df["text"].sample(5, random_state=1):
    print("ДО:   ", t)
    print("ПОСЛЕ:", clean_text(t))
    print()

for t in ["Profit rose 5 % &amp; sales <b>grew</b>", "$ESI on lows, down $1.50", "", None]:
    print(repr(t), "->", repr(clean_text(t)))

ДО:    The loss for the third quarter of 2007 was EUR 0.3 mn smaller than the loss of the second quarter of 2007 .
ПОСЛЕ: the loss for the third quarter of 2007 was eur 0.3 mn smaller than the loss of the second quarter of 2007 .

ДО:    The winner does not have to be present to win .
ПОСЛЕ: the winner does not have to be present to win .

ДО:    $BOBE down $2.84  premarket
ПОСЛЕ: $bobe down $2.84 premarket

ДО:    $MTD (Mettler-Toledo) ready for a breakout
ПОСЛЕ: $mtd (mettler-toledo) ready for a breakout

ДО:    Finnish M-real 's Kyro mill is starting negotiations to cut staff .
ПОСЛЕ: finnish m-real 's kyro mill is starting negotiations to cut staff .

'Profit rose 5 % &amp; sales <b>grew</b>' -> 'profit rose 5 % & sales grew'
'$ESI on lows, down $1.50' -> '$esi on lows, down $1.50'
'' -> ''
None -> ''


Почему не использую агрессивную очистку (лемматизация + удаление стоп-слов и цифр)

In [4]:
example = "Operating profit did not rise and fell 5 % in Q3"
print("лёгкая:     ", clean_text(example))
print("агрессивная:", preprocess_text(example))

лёгкая:      operating profit did not rise and fell 5 % in q3
агрессивная: operating profit rise fell


Подготовка всего датасета: конфликтные метки, дубли, пустые тексты, метки классов, признаки

In [5]:
df_clean = prepare_dataset(df)

print("было:", len(df), "-> стало:", len(df_clean), "удалено:", len(df) - len(df_clean))
print(df_clean["sentiment"].value_counts())
print((df_clean["sentiment"].value_counts(normalize=True) * 100).round(1))
df_clean[["text", "text_clean", "sentiment", "label", "dollar_count", "number_count"]].head()

было: 5842 -> стало: 4803 удалено: 1039
sentiment
neutral     2610
positive    1850
negative     343
Name: count, dtype: int64
sentiment
neutral     54.3
positive    38.5
negative     7.1
Name: proportion, dtype: float64


,text,text_clean,sentiment,label,dollar_count,number_count
0,The GeoSolutions technology will leverage Bene...,the geosolutions technology will leverage bene...,positive,2,0,0
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...","$esi on lows, down $1.50 to $2.50 bk a real po...",negative,0,3,2
2,"For the last quarter of 2010 , Componenta 's n...","for the last quarter of 2010 , componenta 's n...",positive,2,0,4
3,According to the Finnish-Russian Chamber of Co...,according to the finnish-russian chamber of co...,neutral,1,0,0
4,The Swedish buyout firm has sold its remaining...,the swedish buyout firm has sold its remaining...,neutral,1,0,1


In [6]:
df_clean.groupby("sentiment")[["word_count_clean", "dollar_count", "number_count"]].mean().round(2)

,word_count_clean,dollar_count,number_count
sentiment,,,
negative,11.88,0.96,0.97
neutral,21.72,0.05,1.17
positive,19.82,0.41,1.56


In [7]:
df_clean.to_csv("data/financial_clean.csv", index=False)

## Выводы

- предобработка вынесена в src/finnews_sentiment/features/preprocess.py
- очистка лёгкая: html, ссылки, регистр, пробелы. цифры, %, $ и стоп-слова оставляю -
  без них теряется тональность ("did not rise and fell 5 %" -> "rise fell")
- удалены тексты с конфликтными метками и дубли: было 5842, стало 4803, удалено 1039
- почти все конфликты - пары negative/neutral: одно предложение размечено то как негатив,
  то как нейтральное. граница между этими классами размыта даже для разметчиков
- после очистки негатив сократился с ~15% до 7%: neutral 54.3%, positive 38.5%, negative 7.1%
  -> сильный дисбаланс, основная метрика macro F1, в улучшениях попробовать веса классов
- метки: negative=0, neutral=1, positive=2
- признаки из README: word_count_clean, char_count, dollar_count (+ number_count, ticker_count)
- в README dollar_count считался через count('$') - это регулярка "конец строки",
  исправила на count(r"\$")